In [280]:
import os
import sys

sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

In [1]:
tags = ' tag-a tag-b tag-c, tag-a'

In [2]:
tag_list = tags.replace(',', ' ').strip().split()

In [3]:
tag_list = list(set(tag_list))

In [4]:
from datetime import datetime

In [5]:
from models import Bookmark

In [6]:
bm = Bookmark(
    url='https://example.com', title='Example Bookmark', created_at='2025-01-01'
)

In [7]:
bm1 = Bookmark(
    href='https://example.com', description='Example Bookmark', time='2025-01-01'
)

In [8]:
bm1.model_dump()

{'url': HttpUrl('https://example.com/'),
 'title': 'Example Bookmark',
 'created_at': datetime.datetime(2025, 1, 1, 0, 0),
 'note': '',
 'hash': '',
 'tags': [],
 'public': False,
 'toread': False}

In [9]:
bm.model_dump()

{'url': HttpUrl('https://example.com/'),
 'title': 'Example Bookmark',
 'created_at': datetime.datetime(2025, 1, 1, 0, 0),
 'note': '',
 'hash': '',
 'tags': [],
 'public': False,
 'toread': False}

In [1]:
from config import LH_API_ENDPOINTS

In [6]:
LH_API_ENDPOINTS['BOOKMARK_CREATE']

<LH_API_ENDPOINTS.BOOKMARK_CREATE: '/v1/posts/add'>

# HTTPX Response Data Model

In [5]:
import httpx

In [6]:
firefox_user_agent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:139.0) Gecko/20100101 Firefox/139.0'

In [7]:
headers = {
    'User-Agent': firefox_user_agent,
}

## HTTPX GET Response

fetch page title, description [additionally github tags] for bookmark metadata

In [8]:
response = httpx.get(
    url='https://lucumr.pocoo.org/2025/6/12/agentic-coding/',
    timeout=10,
    headers=headers,
    params=None,
)
# response = httpx.get(url="https://github.com/menloresearch/jan", timeout=10, headers=headers)
# response = httpx.get(url="https://realpython.com/python-pydantic/", timeout=10, headers=headers)

In [267]:
response.__dict__

{'status_code': 200,
 'headers': Headers({'server': 'nginx', 'date': 'Wed, 18 Jun 2025 12:02:10 GMT', 'content-type': 'text/html', 'last-modified': 'Thu, 12 Jun 2025 14:46:56 GMT', 'transfer-encoding': 'chunked', 'connection': 'keep-alive', 'content-encoding': 'gzip'}),
 '_request': <Request('GET', 'https://lucumr.pocoo.org/2025/6/12/agentic-coding/')>,
 'next_request': None,
 'extensions': {'http_version': b'HTTP/1.1',
  'reason_phrase': b'OK',
  'network_stream': <httpcore._backends.sync.SyncStream at 0x10d6f87d0>},
 'history': [],
 'is_closed': True,
 'is_stream_consumed': True,
 'default_encoding': 'utf-8',
 'stream': <httpx._client.BoundSyncStream at 0x10e316410>,
 '_num_bytes_downloaded': 10543,
 '_decoder': <httpx._decoders.GZipDecoder at 0x10cf8b910>,
 '_elapsed': datetime.timedelta(microseconds=754575),
 '_content': b'<!doctype html>\n<html lang="en" data-theme="system">\n  <head>\n    <meta charset=utf-8>\n  \n  \n    <title>Agentic Coding Recommendations | Armin Ronacher\'s 

In [268]:
# get the request URL as a string
response.url.__str__()

'https://lucumr.pocoo.org/2025/6/12/agentic-coding/'

In [269]:
# Check if the request was successful
response.status_code == 200

True

In [270]:
# check if the content-type field includes text/html
'text/html' in response.headers.get('content-type', '')

True

In [271]:
response.text

'<!doctype html>\n<html lang="en" data-theme="system">\n  <head>\n    <meta charset=utf-8>\n  \n  \n    <title>Agentic Coding Recommendations | Armin Ronacher\'s Thoughts and Writings</title>\n    <link href="/feed.atom" rel="alternate" title="Armin Ronacher\'s Thoughts and Writings" type="application/atom+xml">\n    <meta name="viewport" content="width=device-width,initial-scale=1">\n    <link rel="preload" href="/static/avatar-tiny.jpg" as="image" type="image/jpeg">\n    <link rel="stylesheet" href="/static/_pygments.css" type="text/css">\n    <link rel="apple-touch-icon" sizes="300x300" href="/static/avatar-small.jpg">\n    <link rel="icon" type="image/png" sizes="300x300" href="/static/avatar-small.jpg">\n    <meta name="twitter:image" content="https://lucumr.pocoo.org/static/avatar-small.jpg">\n    <meta property="og:image" content="https://lucumr.pocoo.org/static/avatar-small.jpg">\n    <link rel="stylesheet" href="/static/fonts.css" type="text/css">\n    <link rel="stylesheet" h

In [272]:
# parse the HTML content using BeautifulSoup
from bs4 import BeautifulSoup

soup: BeautifulSoup = BeautifulSoup(response.text, 'html.parser')

In [274]:
soup.find('title1')

In [257]:
# Many ways to find the title

# 1. find the open graph title
print(f'open-graph title: {soup.find("meta", property="og:title").get("content")}')
# find the open graph description
print(
    f'open-graph description: {soup.find("meta", property="og:description").get("content")}'
)

print('-' * 30)

# 2. Find the title tag
print(f'title: {soup.find("title").text}')

print('-' * 30)

# 3. Find the first h1 tag
print(f'h1: {soup.find("h1").text}')

open-graph title: Agentic Coding Recommendations
open-graph description: Current recommendations of agentic coding.
------------------------------
title: Agentic Coding Recommendations | Armin Ronacher's Thoughts and Writings
------------------------------
h1: Agentic Coding Recommendations


In [259]:
# Additionally for github repos, you can find the topics associated with the repository and use them as tags

gh_tags_elements = soup.find_all('a', attrs={'class': 'topic-tag topic-tag-link'})

# sanitize the html tags to get the text content
gh_tags = [tag.text.strip() for tag in gh_tags_elements if tag.text.strip()]
set(gh_tags)

set()

## HTTPX API Response

In [208]:
api_url = 'https://api.ln.ht/v1/posts/get'

In [209]:
header = {
    'Accept': 'application/json',
    'Authorization': 'Bearer <REDACTED>',
}

In [210]:
payload = {
    'tag': 'stream',
}

In [211]:
response: httpx.Response = httpx.get(
    api_url, headers=header, timeout=10, params=payload
)

In [212]:
response.__dict__

{'status_code': 200,
 'headers': Headers({'date': 'Wed, 18 Jun 2025 10:09:35 GMT', 'server': 'Cowboy', 'cache-control': 'max-age=0, private, must-revalidate', 'content-length': '11287', 'content-type': 'application/json; charset=utf-8', 'x-request-id': 'GEobQgQaUMtMB0QAsQnh', 'strict-transport-security': 'max-age=15768000; includeSubdomains', 'keep-alive': 'timeout=5, max=100', 'connection': 'Keep-Alive'}),
 '_request': <Request('GET', 'https://api.ln.ht/v1/posts/get?tag=stream')>,
 'next_request': None,
 'extensions': {'http_version': b'HTTP/1.1',
  'reason_phrase': b'OK',
  'network_stream': <httpcore._backends.sync.SyncStream at 0x10cd0a850>},
 'history': [],
 'is_closed': True,
 'is_stream_consumed': True,
 'default_encoding': 'utf-8',
 'stream': <httpx._client.BoundSyncStream at 0x10d138710>,
 '_num_bytes_downloaded': 11287,
 '_decoder': <httpx._decoders.IdentityDecoder at 0x10b6d5650>,
 '_elapsed': datetime.timedelta(seconds=1, microseconds=97607),
 '_content': b'{"posts":[{"exte

In [213]:
response.text

'{"posts":[{"extended":"","meta":null,"time":"2025-06-12T07:12:45Z","description":"Stremio Addons","hash":"4b7bf34fa08456fdc050ed1fc5083d13","href":"https://beta.stremio-addons.net/","shared":"yes","tags":"stremio media stream addon","others":0,"toread":"no"},{"extended":"","meta":null,"time":"2025-05-27T08:12:27Z","description":"https://bx-tv.com/","hash":"a6ed0bd97f5b607d8c5f212fc805d180","href":"https://bx-tv.com/","shared":"yes","tags":"stream IPTV media","others":0,"toread":"no"},{"extended":"","meta":null,"time":"2025-05-27T08:10:36Z","description":"Rive","hash":"aeba47173bfc781b9ab9adf06f3c51dd","href":"https://rivestream.org","shared":"yes","tags":"stream media movie tv sport anime IPTV","others":1,"toread":"no"},{"extended":"","meta":null,"time":"2025-01-02T05:25:37Z","description":"webtor - Torrent to Stream / DDL Sites","hash":"59769e321b35e544152d3c41160022fa","href":"https://webtor.io/","shared":"yes","tags":"stream torrent","others":0,"toread":"no"},{"extended":"","meta":

In [180]:
# check if the request was successful
response.status_code == 200

True

In [181]:
# check if the content-type field includes application/json
'application/json' in response.headers.get('content-type', '')

True

In [182]:
response.json()

{'posts': [{'extended': '',
   'meta': None,
   'time': '2025-06-12T07:12:45Z',
   'description': 'Stremio Addons',
   'hash': '4b7bf34fa08456fdc050ed1fc5083d13',
   'href': 'https://beta.stremio-addons.net/',
   'shared': 'yes',
   'tags': 'stremio media stream addon',
   'others': 0,
   'toread': 'no'},
  {'extended': '',
   'meta': None,
   'time': '2025-05-27T08:12:27Z',
   'description': 'https://bx-tv.com/',
   'hash': 'a6ed0bd97f5b607d8c5f212fc805d180',
   'href': 'https://bx-tv.com/',
   'shared': 'yes',
   'tags': 'stream IPTV media',
   'others': 0,
   'toread': 'no'},
  {'extended': '',
   'meta': None,
   'time': '2025-05-27T08:10:36Z',
   'description': 'Rive',
   'hash': 'aeba47173bfc781b9ab9adf06f3c51dd',
   'href': 'https://rivestream.org',
   'shared': 'yes',
   'tags': 'stream media movie tv sport anime IPTV',
   'others': 1,
   'toread': 'no'},
  {'extended': '',
   'meta': None,
   'time': '2025-01-02T05:25:37Z',
   'description': 'webtor - Torrent to Stream / DDL S

In [189]:
from pydantic import BaseModel, Field


class APIResponse(BaseModel):
    """Base model for API responses."""

    status: int = Field(..., description='The status of the API response')
    content: dict[str, list[dict]]

In [190]:
APIResponse(status=response.status_code, content=response.json())

APIResponse(status=200, content={'posts': [{'extended': '', 'meta': None, 'time': '2025-06-12T07:12:45Z', 'description': 'Stremio Addons', 'hash': '4b7bf34fa08456fdc050ed1fc5083d13', 'href': 'https://beta.stremio-addons.net/', 'shared': 'yes', 'tags': 'stremio media stream addon', 'others': 0, 'toread': 'no'}, {'extended': '', 'meta': None, 'time': '2025-05-27T08:12:27Z', 'description': 'https://bx-tv.com/', 'hash': 'a6ed0bd97f5b607d8c5f212fc805d180', 'href': 'https://bx-tv.com/', 'shared': 'yes', 'tags': 'stream IPTV media', 'others': 0, 'toread': 'no'}, {'extended': '', 'meta': None, 'time': '2025-05-27T08:10:36Z', 'description': 'Rive', 'hash': 'aeba47173bfc781b9ab9adf06f3c51dd', 'href': 'https://rivestream.org', 'shared': 'yes', 'tags': 'stream media movie tv sport anime IPTV', 'others': 1, 'toread': 'no'}, {'extended': '', 'meta': None, 'time': '2025-01-02T05:25:37Z', 'description': 'webtor - Torrent to Stream / DDL Sites', 'hash': '59769e321b35e544152d3c41160022fa', 'href': 'http

## fetch link meta

In [1]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
sys.path

['/Users/shubxam/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python311.zip',
 '/Users/shubxam/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11',
 '/Users/shubxam/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/lib-dynload',
 '',
 '/Users/shubxam/Code/linkhut-lib/.venv/lib/python3.11/site-packages',
 '/Users/shubxam/Code/linkhut-lib/src',
 '/Users/shubxam/Code/linkhut-lib/src']

ImportError: attempted relative import with no known parent package

In [9]:
from enum import Enum


class LH_API_ENDPOINTS(Enum):
    """Enum-like class for LinkHut API endpoints."""

    BOOKMARK_GET = '/v1/posts/get'
    BOOKMARK_RECENT = '/v1/posts/recent'
    BOOKMARK_CREATE = '/v1/posts/add'
    BOOKMARK_DELETE = '/v1/posts/delete'
    TAG_SUGGEST = '/v1/posts/suggest'
    TAG_DELETE = '/v1/tags/delete'
    TAG_RENAME = '/v1/tags/rename'

In [13]:
LH_API_ENDPOINTS.BOOKMARK_CREATE.value

'/v1/posts/add'

In [22]:
name = 'linkhut/lib'

In [23]:
if not name.replace('-', '').replace('_', '').isalnum():
    raise ValueError(
        f'Invalid name: {name}. Name must be alphanumeric or contain hyphens/underscores.'
    )

ValueError: Invalid name: linkhut/lib. Name must be alphanumeric or contain hyphens/underscores.

In [26]:
if not []:
    raise ValueError('No tags provided. Please provide at least one tag.')

ValueError: No tags provided. Please provide at least one tag.

In [36]:
import datetime

In [6]:
datetime.datetime.now().isoformat()

'2025-06-19T12:38:58.422851'

In [8]:
# convert to UTC
datetime.datetime.astimezone(datetime.UTC)

TypeError: descriptor 'astimezone' for 'datetime.datetime' objects doesn't apply to a 'datetime.timezone' object

In [14]:
datetime.UTC

datetime.timezone.utc

In [24]:
datetime.datetime.fromisoformat(datetime.date.today().isoformat())

datetime.datetime(2025, 6, 19, 0, 0)

In [34]:
isinstance(datetime.date.today(), datetime.date)

AttributeError: 'method_descriptor' object has no attribute 'today'

In [37]:
date_str = '2025-01-01'
date_date = datetime.date.today()
date_datetime = datetime.datetime.now()

In [38]:
datetime.datetime.fromisoformat(date_str)

datetime.datetime(2025, 1, 1, 0, 0)

In [ ]:
datetime.datetime.fromisoformat(date_date.isoformat()).astimezone(datetime.UTC)

datetime.datetime(2025, 6, 18, 18, 30, tzinfo=datetime.timezone.utc)